# Neo4j dump → Machine shards (W19)

This notebook turns a Neo4j **`.dump`** into snapshots the Machine boots, on Colab, so the
laptop never has to run Neo4j. It does five things, each in its own cell so a failure is
visible where it happens:

1. get the dump into the session (upload, or copy from Drive);
2. install a Neo4j Community tarball of the **same major version the dump came from**,
   with the bundled APOC core jar;
3. `neo4j-admin … load` the dump, start the server, and export everything with
   `apoc.export.csv.all` — one CSV whose header is the union of every property;
4. split that CSV into the node/relationship files the importers read
   (`importers/split_apoc_export.py`), then shard by first label
   (`importers/sharded_import.py`) — one bootable snapshot per label, no dangling edges;
5. zip the shards for download (or copy to Drive).

**Read before running.**

* The Machine's ingest ceiling is memory, roughly **10 KB of resident set per edge**
  (`SWARM_EVOLUTION_DAG.md` §6). A free Colab session has ~12 GB, so a *shard* above
  ~800 k edges will not build here; the label histogram printed in step 4 tells you
  before step 5 starts.
* A 4.x dump loads into Neo4j 4.4 directly. A 4.x dump loaded into 5.x needs the extra
  `database migrate` step the cell offers; a 5.x dump cannot be loaded into 4.4 at all.
  When in doubt, match the version in `NEO4J_VERSION` to the server that made the dump.
* Nothing here was executed by its author against a real dump: the repository's sandbox has
  neither Neo4j nor a route to `dist.neo4j.org`. Every command follows the documented
  tooling, and the Python half is exercised by `validation/test32_neo4j_export_to_shards.py`
  on a synthetic export. The first real run is the trial — if a cell fails, the message is
  the deliverable; paste it back into the repository's issue tracker or the coordinator's
  session.

In [ ]:
#@title Parameters
DUMP_PATH = "/content/neo4j.dump"          #@param {type:"string"}
# DRIVE_DUMP: optional, e.g. /content/drive/MyDrive/graphs/neo4j.dump (leave empty to upload)
DRIVE_DUMP = ""                            #@param {type:"string"}
NEO4J_VERSION = "5.26.0"                   #@param ["5.26.0", "4.4.42"] {allow-input: true}
DATABASE = "neo4j"                         #@param {type:"string"}
# MIGRATE_AFTER_LOAD: True only for a 4.x dump loaded into a 5.x server
MIGRATE_AFTER_LOAD = False                 #@param {type:"boolean"}
HEAP = "4G"                                #@param {type:"string"}
REPO = "https://github.com/StrangeTcy/cat_theo_machine"
BRANCH = "master"                          #@param {type:"string"}
OUT_DIR = "/content/shards"                #@param {type:"string"}

NEO4J_HOME = "/content/neo4j"
MAJOR = int(NEO4J_VERSION.split(".")[0])
print("Neo4j %s (major %d), database %r, shards -> %s" % (NEO4J_VERSION, MAJOR, DATABASE, OUT_DIR))

In [ ]:
#@title 1. The dump: upload it, or copy it from Drive
import os, shutil
if DRIVE_DUMP:
    from google.colab import drive
    drive.mount("/content/drive")
    shutil.copyfile(DRIVE_DUMP, DUMP_PATH)
elif not os.path.exists(DUMP_PATH):
    from google.colab import files
    uploaded = files.upload()  # pick the .dump; large files are faster via Drive
    name = next(iter(uploaded))
    shutil.move(name, DUMP_PATH)
print("dump: %s (%.1f MB)" % (DUMP_PATH, os.path.getsize(DUMP_PATH) / 1e6))

In [ ]:
#@title 2. Neo4j Community + bundled APOC (no auth, export to file enabled)
import os, shutil, subprocess, glob, textwrap
java = "openjdk-17-jre-headless" if MAJOR >= 5 else "openjdk-11-jre-headless"
subprocess.run("apt-get -qq update && apt-get -qq install -y %s > /dev/null" % java, shell=True, check=True)
tarball = "neo4j-community-%s-unix.tar.gz" % NEO4J_VERSION
if not os.path.isdir(NEO4J_HOME):
    subprocess.run(["wget", "-q", "https://dist.neo4j.org/" + tarball], check=True)
    subprocess.run(["tar", "xzf", tarball, "-C", "/content"], check=True)
    os.rename("/content/neo4j-community-%s" % NEO4J_VERSION, NEO4J_HOME)
# APOC core ships inside the distribution since 4.x: copy it into plugins/.
for jar in glob.glob(os.path.join(NEO4J_HOME, "labs", "apoc-*-core.jar")):
    shutil.copy(jar, os.path.join(NEO4J_HOME, "plugins"))
with open(os.path.join(NEO4J_HOME, "conf", "apoc.conf"), "w") as handle:
    handle.write("apoc.export.file.enabled=true\napoc.import.file.enabled=true\n")
conf_path = os.path.join(NEO4J_HOME, "conf", "neo4j.conf")
if MAJOR >= 5:
    extra = textwrap.dedent('''
        dbms.security.auth_enabled=false
        server.memory.heap.initial_size=%s
        server.memory.heap.max_size=%s
        server.default_listen_address=127.0.0.1
        dbms.security.procedures.unrestricted=apoc.*
        dbms.security.procedures.allowlist=apoc.*
    ''' % (HEAP, HEAP))
else:
    extra = textwrap.dedent('''
        dbms.security.auth_enabled=false
        dbms.memory.heap.initial_size=%s
        dbms.memory.heap.max_size=%s
        dbms.default_listen_address=127.0.0.1
        dbms.security.procedures.unrestricted=apoc.*
        dbms.security.procedures.allowlist=apoc.*
    ''' % (HEAP, HEAP))
with open(conf_path, "a") as handle:
    handle.write(extra)
print(subprocess.run([os.path.join(NEO4J_HOME, "bin", "neo4j"), "version"], capture_output=True, text=True).stdout)
print("plugins:", os.listdir(os.path.join(NEO4J_HOME, "plugins")))

In [ ]:
#@title 3a. Load the dump
import os, shutil, subprocess
admin = os.path.join(NEO4J_HOME, "bin", "neo4j-admin")
if MAJOR >= 5:
    # 5.x wants a directory holding <database>.dump
    dumps = "/content/dumps"
    os.makedirs(dumps, exist_ok=True)
    shutil.copyfile(DUMP_PATH, os.path.join(dumps, DATABASE + ".dump"))
    cmd = [admin, "database", "load", DATABASE, "--from-path=" + dumps, "--overwrite-destination=true"]
else:
    cmd = [admin, "load", "--from=" + DUMP_PATH, "--database=" + DATABASE, "--force"]
print(" ".join(cmd))
done = subprocess.run(cmd, capture_output=True, text=True)
print(done.stdout[-3000:], done.stderr[-3000:])
done.check_returncode()
if MIGRATE_AFTER_LOAD and MAJOR >= 5:
    migrate = subprocess.run([admin, "database", "migrate", DATABASE], capture_output=True, text=True)
    print(migrate.stdout[-3000:], migrate.stderr[-3000:])
    migrate.check_returncode()

In [ ]:
#@title 3b. Start the server and export everything to one CSV
import os, subprocess, time
neo4j = os.path.join(NEO4J_HOME, "bin", "neo4j")
shell = os.path.join(NEO4J_HOME, "bin", "cypher-shell")
subprocess.run([neo4j, "start"], check=True)
for _ in range(90):
    probe = subprocess.run([shell, "-d", DATABASE, "RETURN 1;"], capture_output=True, text=True)
    if probe.returncode == 0:
        break
    time.sleep(2)
else:
    print(open(os.path.join(NEO4J_HOME, "logs", "neo4j.log")).read()[-4000:])
    raise SystemExit("Neo4j did not come up; the log tail is above")
counts = subprocess.run([shell, "-d", DATABASE, "--format", "plain",
    "MATCH (n) WITH count(n) AS nodes MATCH ()-[r]->() RETURN nodes, count(r) AS relationships;"],
    capture_output=True, text=True)
print("in the database:", counts.stdout.strip())
export = subprocess.run([shell, "-d", DATABASE, "--format", "plain",
    'CALL apoc.export.csv.all("all.csv", {}) YIELD file, nodes, relationships, properties, time RETURN file, nodes, relationships, properties, time;'],
    capture_output=True, text=True)
print(export.stdout, export.stderr)
export.check_returncode()
EXPORT_CSV = os.path.join(NEO4J_HOME, "import", "all.csv")
print("export: %s (%.1f MB)" % (EXPORT_CSV, os.path.getsize(EXPORT_CSV) / 1e6))

In [ ]:
#@title 4. The repository, and the split (prints the label histogram = the shards to come)
import json, os, subprocess
if not os.path.isdir("/content/cat_theo_machine"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, "/content/cat_theo_machine"], check=True)
subprocess.run(["pip", "-q", "install", "pyyaml", "gmpy2"], check=True)
env = dict(os.environ, PYTHONPATH="/content")
split = subprocess.run(["python", "-m", "cat_theo_machine.importers.split_apoc_export", "/content/split", EXPORT_CSV],
                       cwd="/content", env=env, capture_output=True, text=True)
print(split.stdout, split.stderr)
split.check_returncode()
summary = json.load(open("/content/split/split.json"))
print("node rows %d, relationship rows %d, skipped %d" % (summary["node_rows"], summary["relationship_rows"], summary["skipped_rows"]))
print("first labels (each becomes one shard):")
for label, count in summary["first_labels"].items():
    print("  %-30s %8d" % (label or "(unlabelled)", count))
print("relationship types:", summary["relationship_types"])
biggest = max(summary["first_labels"].values()) if summary["first_labels"] else 0
print("\nrule of thumb: a shard needs ~10 KB per edge while building; the largest label has %d nodes" % biggest)

In [ ]:
#@title 5. Shard (boots the packs once, ~20 s, then one snapshot per label)
import json, os, subprocess
env = dict(os.environ, PYTHONPATH="/content")
proc = subprocess.Popen(["python", "-u", "-m", "cat_theo_machine.importers.sharded_import",
                         "/content/split/nodes.csv", "/content/split/edges.csv", OUT_DIR],
                        cwd="/content", env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    if line.startswith(("SHARD", "SHARDED")) or "Traceback" in line or "Error" in line:
        print(line.rstrip())
proc.wait()
if proc.returncode:
    raise SystemExit("sharded_import exited %d" % proc.returncode)
manifest = json.load(open(os.path.join(OUT_DIR, "manifest.json")))
print(json.dumps({k: v for k, v in manifest.items() if k != "shards"}, indent=2))
for shard in manifest["shards"]:
    print("  %-30s nodes=%-8s edges=%-8s %.1f MB" % (shard["shard"], shard["nodes"], shard["edges"], shard["bytes"] / 1e6))

In [ ]:
#@title 6. Package the shards (download, or copy to Drive)
import os, shutil, subprocess
archive = shutil.make_archive("/content/machine-shards", "zip", OUT_DIR)
print("%s (%.1f MB)" % (archive, os.path.getsize(archive) / 1e6))
if DRIVE_DUMP:
    target = os.path.join(os.path.dirname(DRIVE_DUMP), "machine-shards.zip")
    shutil.copyfile(archive, target)
    print("copied to", target)
else:
    from google.colab import files
    files.download(archive)

## Back on the laptop

Unzip into a directory outside the repository and boot a shard the way any snapshot boots:

```python
from cat_theo_machine import main as MA
from cat_theo_machine import runtime as RT
runtime = RT.boot_from_snapshot("/path/to/shards/Person.json", MA._runtime_namespace())
graph = runtime.graph          # graph.nodes / graph.edges are the imported chains
```

`manifest.json` lists every shard with machine-counted node and edge totals; an edge whose
other end lives in another shard points at a **stub** node there, and the stub's `stub_of`
property edge names the shard that holds the real one (see `importers/sharded_import.py`).
Reasoning over imported content costs ~9 ms per edge per rewrite pass
(`SWARM_EVOLUTION_DAG.md` §6, ingest path) — that, not storage, is the bottleneck to plan for.